# MIE 690A Week 5-6 Project Setup

Run this notebook before opening a project-track notebook. It checks that your Week-4 data are present, reproduces one simple baseline, and writes a small project configuration file.

## Put these files together

- `P0_Project_Setup.ipynb`
- `w4utils.py`
- `w5_common.py`
- `cavity_data.npz` from Week 4 (or the fixed reference copy supplied in the track ZIP)

You are **not** being asked to rewrite the CFD solver. Your Week-5 task begins from the fixed, quality-checked dataset.

In [ ]:
from pathlib import Path
import json, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import w4utils, w5_common
importlib.reload(w4utils); importlib.reload(w5_common)
print("w4utils:",w4utils.W4_UTILS_VERSION)
print("w5_common:",w5_common.W5_COMMON_VERSION)
print("dataset SHA-256:",w4utils.sha256_file("cavity_data.npz"))

data=w5_common.require_week4_files("cavity_data.npz")
print("Available Reynolds numbers:",data["Re"])
print("Stored fields:",[k for k in ("u","v","p","psi","omega") if k in data])
print("Field shape:",data["u"].shape)

## Dataset audit

The table below confirms which cases were labeled as training and blind test cases in Week 4. Your project may define a new controlled split, but the split must be declared **before** test results are inspected.

In [ ]:
audit=pd.DataFrame({
    "Re":data["Re"],"Week4_split":data["split"],
    "accepted":data.get("accepted",np.ones(len(data["Re"]),dtype=bool)),
    "final_residual":data.get("final_residual",np.full(len(data["Re"]),np.nan))
})
display(audit)
assert np.all(np.isfinite(data["u"])) and np.all(np.isfinite(data["v"]))
assert np.allclose(data["p"].mean(axis=(1,2)),0,atol=1e-8)

## Reproduce a non-neural baseline

This is a recovery check, not your final project. We interpolate the two neighboring Week-4 training fields to predict the withheld case at Re=275 and compute the same numerical and physical metrics used in Week 4.

The wall metrics used in this project exclude the two moving-lid corners, where the lid and side-wall velocity conditions meet discontinuously. Therefore, the CFD truth should have essentially zero wall error.

In [ ]:
train_re=data["Re"][data["split"].astype(str)=="train"]
pred=w5_common.interpolate_case(data,275,train_re)
report=w5_common.evaluate_prediction(data,275,pred)
display(pd.DataFrame([{"method":"field interpolation","Re":275,**report}]))
fig=w5_common.plot_case_evidence(data,275,{"interpolation":pred},"Setup baseline")
plt.show()
truth_idx=int(np.where(data["Re"]==275)[0][0])
truth=(data["u"][truth_idx],data["v"][truth_idx],data["p"][truth_idx])
truth_report=w5_common.evaluate_prediction(data,275,truth)
display(pd.DataFrame([{"method":"CFD truth reference","Re":275,**truth_report}]))

## Select one project variant

Choose exactly one of the ten variants listed in the guide. You may change this choice only before the Week-5 checkpoint.

In [ ]:
# EDIT THESE THREE LINES.
STUDENT_NAME = "Your Name"
PROJECT_VARIANT = "1A"   # one of 1A,1B,2A,2B,3A,3B,4A,4B,5A,5B
ONE_SENTENCE_QUESTION = "Replace this with your research question."

card={"student":STUDENT_NAME,"variant":PROJECT_VARIANT,
      "research_question":ONE_SENTENCE_QUESTION,
      "dataset":"cavity_data.npz","dataset_cases":data["Re"].tolist()}
w5_common.write_project_card("project_choice.json",card)
print(json.dumps(card,indent=2))

## Submit at the beginning of Week 5

- `project_choice.json`
- one screenshot of the dataset audit
- the interpolation baseline metric table
- your selected project notebook

Do not continue until the baseline runs and you can explain what its error metrics mean.